[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week04/week04-mission-analysis.ipynb)

# 미션 분석 입문 — 궤도요소, 태양동기궤도, 궤적설계·GNC 개관
### Introduction to Mission Analysis — Orbital Elements, SSO, Trajectory Design & GNC

**우주수송정책과 발사체 기술**
> 📎 본 노트북은 4주차 정규 강의자료(`week04.ipynb`, 발사체 시스템 설계와 운용)의 **보충 자료(supplementary material)**입니다. 원본 강의(`On the Mission Analysis`)의 목차·전개를 따르되, 정적 슬라이드 대신 슬라이더로 직접 조작하는 인터랙티브 시각화로 재구성했습니다.

발사체가 "얼마나 빨리, 어느 방향으로, 무엇을 향해" 날아야 하는가를 정하는 작업이 미션 분석(Mission Analysis)입니다. 4주차 본편이 상승궤적(ascent trajectory)과 GNC의 개념을 다뤘다면, 본 보충자료는 그 상승이 도달해야 할 **목표 궤도를 어떻게 숫자로 정의하는지**(궤도 6요소), **왜 특정 궤도가 정찰·관측 임무에 유리한지**(태양동기궤도), 그리고 **궤적을 어떻게 최적화하고 유도·항법·제어로 실현하는지**를 한 단계 더 파고듭니다.

## 목차

1. 인공위성 궤도의 6요소 — 크기·모양·방향·위치를 숫자로 정의하기
2. 구면조화함수와 르장드르 전개 — 지구는 완전한 구가 아니다
3. 태양동기궤도(SSO)와 J2 섭동 — 편평한 지구가 만드는 세차운동의 활용
4. 위성의 정찰·통신 영역(Footprint) — 고도와 앙각이 정하는 커버리지
5. 발사체 궤적과 손실 — 목표 궤도까지 가는 비용 (2·4주차 복습 연계)
6. 궤적 최적화(Trajectory Optimization) — 최적제어 문제로서의 상승궤적
7. 유도·항법·제어(GNC) — 계획을 실행으로 옮기는 삼각형
8. 요약 및 참고문헌

In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")

---
# 01. 인공위성 궤도의 6요소
### Six Classical Orbital Elements

*CH.01 궤도 6요소*
## 궤도를 숫자 6개로 완전히 표현하기

- 뉴턴 역학에서 2체문제(two-body problem)의 해는 항상 원뿔곡선(원·타원·포물선·쌍곡선)
- 이 원뿔곡선의 **크기·모양·공간상의 방향·궤도상 위치**를 정하면 임의 시각의 위성 상태(위치·속도)가 완전히 결정됨
- 6개의 독립된 수 — **케플러 궤도요소(Keplerian orbital elements)** — 로 구성
  - 궤도의 크기와 모양 (2개) — 장반경 a, 이심률 e
  - 궤도면의 방향 (3개) — 경사각 i, 승교점적경 Ω, 근점편각 ω
  - 위성의 현재 위치 (1개) — 진근점이각 ν
- 위치·속도벡터(6개 성분)와 궤도요소(6개)는 서로 1:1 변환 가능 — 정보량은 동일하고 **해석의 편의**만 다름

*CH.01 궤도 6요소*
## (1) 궤도의 크기와 모양 — 장반경 a, 이심률 e

- **장반경(Semi-major Axis, $a$)** — 타원 궤도의 긴 반지름, 궤도의 전체 '크기'와 주기를 결정
  - 케플러 제3법칙: $T = 2\pi\sqrt{a^3/\mu}$ — 주기는 오직 $a$에만 의존 (모양과 무관)
- **이심률(Eccentricity, $e$)** — 궤도가 원에서 얼마나 벗어났는지, $0\le e<1$(타원), $e=0$(원), $e=1$(포물선), $e>1$(쌍곡선)
- 근지점(perigee)·원지점(apogee) 반경으로 환산:
$$r_p = a(1-e), \qquad r_a = a(1+e)$$
- 발사체 미션 관점 — 순환궤도(e≈0) 투입이 일반적으로 요구 Δv가 크고, 타원 전이궤도(e>0)는 GTO처럼 상단부 임무로 넘기는 절충안

*CH.01 궤도 6요소*
## (2) 궤도면의 방향 — 경사각 i, 승교점적경 Ω, 근점편각 ω

- **경사각(Inclination, $i$)** — 궤도면과 지구 적도면 사이의 각. $i=0°$ 적도궤도, $i=90°$ 극궤도, $i>90°$ 역행(retrograde)궤도
  - 발사장 위도가 $i$의 하한을 결정 — 위도보다 낮은 경사각은 방위각 조정만으로 도달 불가(도그레그 필요)
- **승교점적경(Right Ascension of the Ascending Node, $\Omega$, RAAN)** — 위성이 남→북으로 적도면을 통과하는 점(승교점)의, 춘분점 기준 방위
- **근점편각(Argument of Perigee, $\omega$)** — 승교점에서 근지점까지, 궤도면 내에서 잰 각
- 세 각이 함께 궤도면이 3차원 공간에서 "어느 쪽을 향해 기울어 있는지"를 완전히 고정

*CH.01 궤도 6요소*
## (3) 위성의 현재 위치 — 진근점이각 ν

- **진근점이각(True Anomaly, $\nu$)** — 근지점을 기준으로, 궤도면 내에서 위성이 현재 어디에 있는지를 나타내는 각
- 궤도 모양·방향(5개 요소)이 "궤도라는 트랙 자체"를 정의한다면, $\nu$는 "그 트랙 위 몇 바퀴째 어느 지점"을 정의
- 시간에 따라 $\nu$만 변화(케플러 방정식을 따라 비균일하게 변화, $e=0$이면 등각속도) — 나머지 5개는 섭동이 없다면 일정
- 아래 위젯에서 6개 슬라이더를 모두 조작하며 "크기·모양 2개 + 방향 3개 + 위치 1개" 구조를 직접 체감

In [ ]:
orbital_elements_explorer()

---
# 02. 구면조화함수와 르장드르 전개
### Spherical Harmonics & Legendre Expansion

*CH.02 르장드르 전개*
## 복잡한 함수를 단순한 함수의 합으로 — 기저함수라는 아이디어

- **푸리에 급수** — 복잡한 주기함수를 단순한 사인·코사인 함수의 선형결합으로 분해
- 대수학의 기본 정리를 함수공간으로 확장한 직관:
  > "내적이 정의된 함수공간에서는 어떠한 함수(벡터)라도 기저함수(벡터)의 선형결합으로 표현할 수 있다"
- 즉 사인·코사인이 '주기함수 공간'의 기저벡터 역할 — 이 아이디어가 지구 중력장 표현에도 그대로 적용됨
- **르장드르 급수** — 복잡한 함수 $f(x)$를 르장드르 다항식 $P_k(x)$의 선형결합으로 표현
  - $-1<x<1$ 구간에서 정의, 푸리에 급수와 마찬가지로 직교성을 이용해 전개계수 결정

*CH.02 르장드르 전개*
## 지구 중력 포텐셜의 구면조화 전개

- 실제 지구는 완전한 구가 아니라 **편평한 회전타원체(oblate spheroid)**이며 내부 질량분포도 불균일
- 따라서 중력 포텐셜 $V(r,\Phi)$를 구면조화함수(르장드르 다항식 기반)로 전개해 근사:
$$V(r,\Phi) = \frac{\mu}{r}\left[1 - \sum_{k=2}^{\infty} J_k \left(\frac{R_e}{r}\right)^k P_k(\sin\Phi)\right]$$
- $J_k$: 구역 조화항(zonal harmonics) 계수, $R_e$: 지구 적도 반지름, $\Phi$: 지구중심 기준 위도, $P_k$: $k$차 르장드르 다항식
- 가장 지배적인 항이 **$J_2$**(지구 편평도): $J_2 = 1.08263\times10^{-3}$
- $J_2$가 만드는 비대칭 중력장 → 궤도면에 지속적인 토크 → **승교점의 세차운동(RAAN drift)** 발생
- 이 "결점(defect)"을 역이용하는 것이 다음 절의 태양동기궤도(SSO)

---
# 03. 태양동기궤도(SSO)와 J2 섭동
### Sun-Synchronous Orbit & J2 Perturbation

*CH.03 SSO*
## J2가 만드는 승교점 세차 — 버그를 기능으로

- 위성 궤도의 승교점(RAAN)은 여러 섭동력으로 서서히 이동하는데, 그중 **지구 편평성에 의한 $J_2$ 섭동력**이 압도적 주요 원인
- 평균 세차율(원궤도 근사):
$$\dot{\Omega} \approx -\frac{3}{2} n J_2 \left(\frac{R_e}{a}\right)^2 \cos i \qquad \left(n=\sqrt{\mu/a^3}\right)$$
- $\cos i$ 항 때문에 **역행궤도($i>90°$)일 때 $\dot{\Omega}>0$**(동쪽으로 세차) — 순행궤도에서는 반대 부호
- 지구가 태양 주위를 한 바퀴 도는 데 걸리는 시간(항성년) 기준, 평균 공전각속도 = **0.9856°/day**

*CH.03 SSO*
## 태양동기궤도(SSO)란 무엇이며 왜 쓰는가

- 궤도면의 세차율 $\dot{\Omega}$를 지구 공전각속도(+0.9856°/day)와 정확히 일치시키면, 궤도면-태양 사이의 상대각이 항상 일정하게 유지됨
- 그 결과:
  - 승교선의 지방시(LTAN, Local Time of Ascending Node)가 고정
  - 매 궤도마다 **동일한 지방시에 동일한 위도**를 통과
  - 태양 조도 조건이 일정 → 원격탐사(remote sensing) 관측 영상의 그림자·명암 조건이 일관됨
  - 위성 태양전지판의 지향 제어를 최소화 가능(태양-궤도면 상대각이 거의 고정)
- 대가는 **경사각이 자유변수가 아니라는 점** — 원하는 고도를 정하면 SSO를 만족하는 경사각은 (거의) 하나로 정해짐(통상 96°~100° 부근의 역행궤도)
- 아래 위젯에서 고도를 바꾸며 SSO를 만족시키는 경사각이 어떻게 변하는지, 그리고 경사각을 어긋내면 세차율이 목표치에서 벗어나는 모습을 직접 확인

In [ ]:
sso_explorer()

---
# 04. 위성의 정찰·통신 영역(Footprint)
### Reconnaissance Swath & Communication Footprint

*CH.04 Footprint*
## 위성이 "볼 수 있는" 또는 "닿을 수 있는" 지표 영역

- 위성-지구 기하학은 세 각으로 요약됨
  - $\rho$ — 위성에서 본 지구의 각반경, $\sin\rho = R_e/(R_e+h)$
  - $\varepsilon$ — 지상국(또는 관측대상)에서 본 위성의 최소 앙각(elevation angle) — 임무 요구조건으로 주어짐
  - $\lambda$ — 지구중심각(central angle), $\eta+\lambda+\varepsilon=90°$, $\sin\eta=\sin\rho\cos\varepsilon$
- 지표 커버리지 반경(대권거리): $\;d = R_e\lambda$(라디안)
- **정찰/관측위성(저궤도)** — 이 반경이 한 통과당 얻을 수 있는 관측 스와스(swath)의 기하학적 상한을 결정
- **통신위성(주로 정지궤도)** — 이 반경이 지상국과 동시에 가시선(line-of-sight)을 확보할 수 있는 서비스 영역(footprint)을 결정
- 같은 공식이 고도만 바뀌면 저궤도 정찰위성과 정지궤도 통신위성에 공통으로 적용됨 — 아래 위젯에서 고도를 300km부터 36,000km까지 넓게 훑어가며 확인

In [ ]:
ground_footprint_explorer()

---
# 05. 발사체 궤적과 손실
### Launch Trajectory & Δv Losses (2·4주차 복습 연계)

*CH.05 궤적과 손실*
## 목표 궤도까지 가는 비용 — 이상적 Δv와 손실의 차이

- 궤도 6요소로 목표를 정의했다면, 이제 그 궤도까지 **발사체가 실제로 지불해야 하는 Δv**를 따져야 함
- 발사체 궤적(수직상승 → 중력선회 → 진공궤적 → 투입)은 2·4주차 본편에서 다룬 그대로 — 본 절에서는 그 궤적이 만드는 **손실(loss)** 항목을 미션 분석 관점에서 재정리
  - **중력손실(gravity loss)** — 추력이 중력을 거스르는 방향 성분에 쓰이며 발생, 연소시간이 길고 초기 T/W가 낮을수록 커짐
  - **항력손실(drag loss)** — 대기 중 구간에서 공력저항을 극복하는 데 쓰이는 손실, 저고도 고속구간에서 지배적
  - **조종손실(steering loss)** — 추력벡터가 순간 비행경로각과 어긋나면서 발생하는 손실
  - 지구 자전을 이용하는 **자전 보너스**는 손실의 반대 방향으로 Δv 예산을 절감
- 이 네 요소의 합이 이상적 궤도속도(vis-viva) 위에 얹혀 최종 요구 Δv를 구성 — 2주차 Δv 예산 계산기·4주차 Max-Q 탐색기를 아래에서 다시 불러 궤도요소(6절)와 연결지어 복습

In [ ]:
delta_v_budget_calculator()

In [ ]:
max_q_explorer()

---
# 06. 궤적 최적화(Trajectory Optimization)
### Ascent Trajectory as an Optimal Control Problem

*CH.06 궤적 최적화*
## 상승궤적 설계는 곧 최적제어 문제

- 미션 분석의 마지막 단계는 "이 손실들을 최소화하는 궤적(피치 프로그램)을 어떻게 찾는가" — **최적제어(optimal control)** 문제로 정식화
- 상태변수 — 고도 $h$, 속도, 비행경로각 등 / 제어변수 — 피치각(또는 추력방향), 스로틀
- 목적함수 — 잔여연료 최대화(=Δv 손실 최소화), 제약조건 — 동압(Max-Q) 상한, 축가속도 상한, 최종 궤도조건(목표 6요소 부합)
- 두 계열의 풀이법
  - **간접법(indirect method, shooting)** — 폰트리아긴 최소원리로 오일러-라그랑주 방정식을 유도한 뒤 경계값 문제(BVP)를 풂. 이론적으로 정밀하나 초기추정(공역변수)에 민감
  - **직접법(direct method, collocation)** — 궤적을 유한 개 구간(mesh)으로 이산화해 비선형계획법(NLP)으로 변환, 상태·제어를 동시에 최적화
- **최근 추세** — 컴퓨팅 성능 향상으로 복잡한 제약조건(다단 분리, 페어링 분리, 궤도 제약)을 훨씬 안정적으로 다루는 **직접 콜로케이션 방식**이 더 선호되는 추세
- 발사체 사업기획 관점에서는 궤적최적화 결과가 곧 페이로드 마진(payload margin)의 실질적 상한을 정함 — 캡스톤 ⑤·⑥절 성능/재무 분석과 직결

---
# 07. 유도·항법·제어(GNC)
### Guidance, Navigation & Control

*CH.07 GNC*
## 세 가지 질문에 각각 답하는 세 기능

- **항법(Navigation)** — "나는 지금 어디에, 얼마나 빠르게 있는가?"
  - 측정된 각운동·가속도로부터 자세·속도·위치 벡터를 결정(현재 상태의 측정·계산)
- **유도(Guidance)** — "지금부터 목표 궤도까지 어떤 경로로 가야 하는가?"
  - 발사체의 현재 위치에서 지정된 목표 궤도까지 원하는 방향·궤적을 결정(예측-보정, outer loop)
- **제어(Control)** — "그 경로를 따라가도록 지금 당장 무엇을 해야 하는가?"
  - 유도명령을 추종하면서 자세 안정성을 유지하도록 제어력·모멘트를 생성(안정화, inner loop)
- NGC(Navigation-Guidance-Control) 기능은 비행컴퓨터의 비행 소프트웨어에서 구현·수행됨
- 세 기능은 위계를 이룸 — 항법이 "현재"를 알려주고, 유도가 "목표까지 경로"를 계산하고, 제어가 그 경로를 순간순간 실현. 항법 오차는 유도 오차로, 유도 오차는 제어 부담으로 전파됨
- 4주차 본편에서 다룬 슬로싱·굽힘모드·POGO 같은 연성(coupling) 문제는 대부분 **제어(inner loop)** 단에서 안정성 여유(margin)로 관리

---
# 08. 요약 및 참고문헌
### Summary & References

*CH.08 요약*
## 핵심 요약

- 궤도는 6개의 독립된 수(a, e, i, Ω, ω, ν)로 완전히 정의된다 — 크기·모양 2개, 방향 3개, 위치 1개
- 지구는 완전한 구가 아니다($J_2$) — 이 비대칭이 만드는 RAAN 세차를 역이용한 것이 태양동기궤도(SSO)
- 위성의 정찰/통신 영역(footprint)은 고도와 최소 앙각만으로 기하학적으로 결정된다 — 저궤도 정찰과 정지궤도 통신에 동일한 공식이 적용된다
- 발사체가 목표 궤도에 도달하는 데 드는 Δv는 이상적 값 위에 중력·항력·조종손실이 더해진 값이다
- 상승궤적 설계는 손실을 최소화하는 최적제어 문제이며, 계산성능의 향상으로 직접 콜로케이션 방식이 주류가 되고 있다
- GNC는 항법(현재 상태 파악)·유도(목표까지 경로 계획)·제어(경로 실시간 추종)의 위계적 삼각형이다

---
# 참고문헌

- Vallado, D. A. (2013), *Fundamentals of Astrodynamics and Applications*, 4th ed., Microcosm Press
- Wertz, J. R. & Larson, W. J. (1999), *Space Mission Analysis and Design (SMAD)*, 3rd ed., Microcosm Press
- Karabeyoglu, A., *AA284a Lecture 7: Launch Trajectories*, Stanford University
- Curtis, H. D. (2020), *Orbital Mechanics for Engineering Students*, 4th ed., Butterworth-Heinemann
- Betts, J. T. (2010), *Practical Methods for Optimal Control and Estimation Using Nonlinear Programming*, 2nd ed., SIAM

---
*본 노트북은 4주차 정규 강의자료의 보충 자료입니다. 정규 강의노트는 `week04.ipynb`를 참고하십시오.*